# temporal_lstm_model1

**Spatial:** GAT (fixed)  |  **Temporal:** LSTM  |  **Model:** 1

All shared code cells (1–11) are identical across `temporal_gru_model1.ipynb`, `temporal_lstm_model1.ipynb` and `temporal_tcn_model1.ipynb`.
Only cell 1 (`EXPERIMENT_NAME`) and cell 12 (architecture) differ.

In [ ]:
# user configuration

# kaggle account (lowercase, as on kaggle.com)
KAGGLE_USERNAME    = 'timweischueler'   # <- ADJUST
KAGGLE_MODEL_NAME  = 'temp-lstm-m1'   # <- AJDUST
KAGGLE_VARIANT     = 'default'
EXPERIMENT_NAME    = 'temporal_lstm_model1'
ENABLE_KAGGLE_UPLOAD   = True                 # <- ADJUST
UPLOAD_EVERY_N_EPOCHS  = 5                   # <- ADJUST

MODEL_ID           = 1              # whcih city model (1 or 2)

KAGGLE_HANDLE = f'{KAGGLE_USERNAME}/{KAGGLE_MODEL_NAME}/pyTorch/{KAGGLE_VARIANT}'

print('=== user config ===')
print(f'  Kaggle Handle:    {KAGGLE_HANDLE}')
print(f'  Experiment:       {EXPERIMENT_NAME}')
print(f'  Kaggle Upload:    {ENABLE_KAGGLE_UPLOAD}')
print(f'  Upload interval: every {UPLOAD_EVERY_N_EPOCHS} epochs')

# platform settings
# 'kaggle' = kaggle notebook  |  'colab' = google colab
PLATFORM = 'kaggle'           # <- ADJUST

# google colab setings (only if PLATFORM = 'colab')
# upload unzipped data to Drive: Drive / FloodAI / Data / Models / Model_1 / ...
# Model_2 / ...
DRIVE_DATA_PATH = '/content/drive/MyDrive/ClimateProject/Models'   # <- ADJUST
DRIVE_SAVE_PATH = '/content/drive/MyDrive/ClimateProject/Results'

In [ ]:
# installation
import subprocess, sys

pkgs = [('scipy', 'scipy'), ('torch_geometric', 'torch_geometric')]
if PLATFORM == 'kaggle':
    pkgs.append(('kagglehub', 'kagglehub'))

for pkg, imp in pkgs:
    try:
        __import__(imp)
        print(f'{pkg} OK')
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=True)
        print(f'{pkg} installed')

print('Installation complete.')

In [ ]:
# imports, paths, hyperparameters
import os, copy, time, json as _json
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torch_geometric.nn import GATConv
from scipy.spatial import cKDTree
if PLATFORM == 'kaggle':
    from kagglehub import model_upload

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    torch.backends.cudnn.benchmark = True

if PLATFORM == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    BASE           = Path(DRIVE_DATA_PATH)
    SAVE_DIR       = Path(DRIVE_SAVE_PATH) / f'checkpoints_{EXPERIMENT_NAME}'
    KAGGLE_MDL_DIR = Path(DRIVE_SAVE_PATH) / f'kaggle_models_{EXPERIMENT_NAME}'
    ENABLE_KAGGLE_UPLOAD = False          # no upload from coalb
else:
    _hits = list(Path('/kaggle/input').rglob('1d_nodes_static.csv'))
    if _hits:
        _d = _hits[0].parent
        if _d.name in ('train', 'test'): _d = _d.parent
        BASE = _d.parent
    else:
        BASE = Path('/kaggle/input')
    SAVE_DIR       = Path(f'/kaggle/working/checkpoints_{EXPERIMENT_NAME}')
    KAGGLE_MDL_DIR = Path(f'/kaggle/working/kaggle_models_{EXPERIMENT_NAME}')

SAVE_DIR.mkdir(parents=True, exist_ok=True)
KAGGLE_MDL_DIR.mkdir(parents=True, exist_ok=True)

SEQ_LEN    = 10
PRED_STEPS = 5
STRIDE     = 3
H_DIM      = 32
H_GNN      = 16
BS_1D      = 32
BS_2D      = 4
LR         = 1e-3
EPOCHS     = 100
PATIENCE   = 20
DROP       = 0.25
WD         = 1e-2
EMA_DECAY  = 0.995
NOISE_STD  = 0.02
CHUNK_2D   = 256

MODEL_PATH = BASE / f'Model_{MODEL_ID}'  # explicitly Model_1 or Model_2

print(f'Device: {DEVICE}')
print(f'BASE:   {BASE}')
print(f'SAVE:   {SAVE_DIR}')

In [ ]:
# kaggle model saving

def _upload_checkpoint(model_tag, epoch, rmse, extra_meta=None):
    if not ENABLE_KAGGLE_UPLOAD:
        return
    upload_dir = KAGGLE_MDL_DIR / model_tag
    upload_dir.mkdir(parents=True, exist_ok=True)
    src_pth = SAVE_DIR / f'{model_tag}.pth'
    dst_pth = upload_dir / f'{model_tag}.pth'
    if src_pth.exists():
        import shutil
        shutil.copy2(src_pth, dst_pth)
    meta = {'model_tag': model_tag, 'experiment': EXPERIMENT_NAME,
            'epoch': epoch, 'val_rmse': float(rmse),
            'kaggle_username': KAGGLE_USERNAME}
    if extra_meta:
        meta.update(extra_meta)
    with open(upload_dir / 'meta.json', 'w') as f:
        _json.dump(meta, f, indent=2)
    version_notes = (f'{EXPERIMENT_NAME} | {model_tag} | '
                     f'epoch {epoch} | val_rmse {rmse:.6f}')
    try:
        model_upload(handle=KAGGLE_HANDLE, local_model_dir=str(upload_dir),
                     version_notes=version_notes)
        print(f'    Kaggle upload OK: {version_notes}')
    except Exception as e:
        print(f'    Kaggle upload failed: {e}')

print('Kaggle model saving defined.')
print(f'  Handle: {KAGGLE_HANDLE}')
print(f'  Upload enabled: {ENABLE_KAGGLE_UPLOAD}')

# platform settings
# 'kaggle' = kaggle notebook  |  'colab' = google colab
PLATFORM = 'kaggle'           # <- ADJSUT

# google colab settings (only if PLATFORM = 'colab')
# upload unzipped data to Drive: Drive / FloodAI / Data / Models / Model_1 / ...
# Model_2 / ...
DRIVE_DATA_PATH = '/content/drive/MyDrive/FloodAI/Data/Models'   # <- ADUJST
DRIVE_SAVE_PATH = '/content/drive/MyDrive/FloodAI/Results'        # <- ADJUST


In [ ]:
# data loading helpers

def find_static(model_path, name):
    for sub in ['train', 'test']:
        p = Path(model_path) / sub / name
        if p.exists(): return p
    p = Path(model_path) / name
    if p.exists(): return p
    hits = list(Path(model_path).rglob(name))
    if hits: return hits[0]
    raise FileNotFoundError(f'{name} not found in {model_path}')


def build_knn_graph(xy, k=6, device='cpu'):
    tree = cKDTree(xy)
    _, idx = tree.query(xy, k=k + 1)
    N = len(xy)
    src, dst = [], []
    for i in range(N):
        for j in idx[i, 1:]:
            src.append(i); dst.append(int(j))
    return torch.tensor([src, dst], dtype=torch.long).to(device)


def load_graph_data(model_path, device):
    model_path = Path(model_path)
    n1d  = pd.read_csv(find_static(model_path, '1d_nodes_static.csv'))
    ei1d = pd.read_csv(find_static(model_path, '1d_edge_index.csv'))
    cols1d = [c for c in ['depth', 'invert_elevation', 'surface_elevation', 'base_area']
              if c in n1d.columns]
    nf = n1d[cols1d].fillna(n1d.mean(numeric_only=True)).values.astype('float32')
    nf = (nf - nf.mean(0, keepdims=True)) / (nf.std(0, keepdims=True) + 1e-6)
    NF_1D  = torch.tensor(nf, dtype=torch.float32).to(device)
    ei_raw = torch.tensor([ei1d['from_node'].values, ei1d['to_node'].values], dtype=torch.long)
    EI_1D  = torch.cat([ei_raw, ei_raw.flip(0)], 1).to(device)
    N_1D   = len(n1d)

    n2d  = pd.read_csv(find_static(model_path, '2d_nodes_static.csv'))
    N_2D = len(n2d)
    cols2d = [c for c in ['area', 'roughness', 'min_elevation', 'elevation',
                           'aspect', 'curvature', 'flow_accumulation']
              if c in n2d.columns]
    sf = n2d[cols2d].fillna(n2d.mean(numeric_only=True)).values.astype('float32')
    sf = (sf - sf.mean(0, keepdims=True)) / (sf.std(0, keepdims=True) + 1e-6)
    SF_2D   = torch.tensor(sf, dtype=torch.float32).to(device)
    SDIM_2D = SF_2D.shape[1]
    if 'position_x' in n2d.columns and 'position_y' in n2d.columns:
        xy = n2d[['position_x', 'position_y']].values.astype('float32')
    else:
        xy = sf[:, :2]
        print('  WARNING: position_x/y not found')
    xy_norm = (xy - xy.mean(0)) / (xy.std(0) + 1e-6)
    EI_2D   = build_knn_graph(xy_norm, k=6, device=device)

    print(f'  1D: {N_1D} Nodes, {EI_1D.shape[1]} edges, {NF_1D.shape[1]} features')
    print(f'  2D: {N_2D} Nodes, {EI_2D.shape[1]} kNN-edges, {SDIM_2D} features')
    return (NF_1D, EI_1D, N_1D), (SF_2D, SDIM_2D, N_2D), EI_2D


def load_event(ev_path):
    d1n = pd.read_csv(ev_path / '1d_nodes_dynamic_all.csv')
    d2n = pd.read_csv(ev_path / '2d_nodes_dynamic_all.csv')
    wl1d = d1n.pivot(index='timestep', columns='node_idx', values='water_level').ffill().bfill()
    wl2d = d2n.pivot(index='timestep', columns='node_idx', values='water_level').ffill().bfill()
    rain = d2n.groupby('timestep')['rainfall'].mean().fillna(0)
    for df in [wl1d, wl2d]:
        for c in df.columns:
            if df[c].isna().any(): df[c] = df[c].fillna(df[c].mean())
    mt = min(len(wl1d), len(wl2d), len(rain))
    return {'wl1d': wl1d.values[:mt].astype('float32'),
            'wl2d': wl2d.values[:mt].astype('float32'),
            'rain': rain.values[:mt].astype('float32')}

print('Data loading defined.')

In [ ]:
# train/val/test split
SPLIT_SEED = 2026
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.15, 0.15


def get_event_dirs(model_path):
    train_dir = Path(model_path) / 'train'
    if not train_dir.exists():
        hits = [d for d in Path(model_path).rglob('train') if d.is_dir()]
        train_dir = hits[0] if hits else train_dir
    return sorted(
        [d for d in train_dir.iterdir() if d.is_dir() and d.name.startswith('event_')],
        key=lambda d: int(d.name.split('_')[1])
    )


def get_splits(model_path):
    all_dirs = get_event_dirs(model_path)
    n = len(all_dirs)
    assert n >= 5, f'Need at least 5 events, found: {n}'
    rng = np.random.RandomState(SPLIT_SEED)
    idx = rng.permutation(n)
    n_test  = max(1, int(n * TEST_FRAC))
    n_val   = max(1, int(n * VAL_FRAC))
    n_train = n - n_val - n_test
    train_dirs = [all_dirs[i] for i in sorted(idx[:n_train])]
    val_dirs   = [all_dirs[i] for i in sorted(idx[n_train:n_train + n_val])]
    test_dirs  = [all_dirs[i] for i in sorted(idx[n_train + n_val:])]
    print(f'  {Path(model_path).name}: {len(train_dirs)} train / '
          f'{len(val_dirs)} val / {len(test_dirs)} test')
    return train_dirs, val_dirs, test_dirs

print('Split defined (seed=2026, 70/15/15).')

In [ ]:
# dataset classes

def _rain_extras(rain_norm, rm, rs):
    sl   = len(rain_norm)
    cum  = np.cumsum(rain_norm) / np.arange(1, sl + 1, dtype='float32')
    grad = np.gradient(rain_norm).astype('float32')
    cum  = ((cum  - rm) / (rs + 1e-8)).astype('float32')
    grad = (grad        / (rs + 1e-8)).astype('float32')
    return cum, grad


class DS1D(Dataset):
    def __init__(self, event_dirs, sl=SEQ_LEN, ps=PRED_STEPS, stride=STRIDE, ns=None):
        self.sl, self.ps = sl, ps
        valid = []
        for ev in event_dirs:
            try:
                d = load_event(ev)
                if d['wl1d'].shape[0] > sl + ps + 1: valid.append(d)
            except Exception as e:
                print(f'  Skipping {ev.name}: {e}')
        self.rw = [d['wl1d'] for d in valid]
        self.rr = [d['rain'] for d in valid]
        if ns is None:
            all_wl = np.concatenate(self.rw).flatten()
            all_r  = np.concatenate(self.rr).flatten()
            self.ns = {'wm': float(all_wl.mean()), 'ws': float(max(all_wl.std(), 1.0)),
                       'rm': float(all_r.mean()),  'rs': float(max(all_r.std(), 1e-6))}
        else:
            self.ns = ns
        s = self.ns
        self.wn = [((w - s['wm']) / s['ws']).astype('float32') for w in self.rw]
        self.rn = [((r - s['rm']) / s['rs']).astype('float32') for r in self.rr]
        self.samples = [(i, t) for i, w in enumerate(self.wn)
                        for t in range(1, len(w) - sl - ps, stride)]
        print(f'  DS1D: {len(event_dirs)} Events ({len(valid)} valid) '
              f'-> {len(self.samples)} Samples')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        i, t = self.samples[idx]
        sl, ps, s = self.sl, self.ps, self.ns
        wl   = self.wn[i][t:t + sl]
        dwl  = wl - self.wn[i][t - 1:t + sl - 1]
        rain = self.rn[i][t:t + sl]
        cum_rain, rain_grad = _rain_extras(rain, s['rm'], s['rs'])
        last  = self.rw[i][t + sl - 1]
        delta = ((self.rw[i][t + sl:t + sl + ps] - last) / s['ws']).astype('float32')
        return {'wl':        torch.tensor(wl,        dtype=torch.float32),
                'dwl':       torch.tensor(dwl,       dtype=torch.float32),
                'rain':      torch.tensor(rain,      dtype=torch.float32),
                'cum_rain':  torch.tensor(cum_rain,  dtype=torch.float32),
                'rain_grad': torch.tensor(rain_grad, dtype=torch.float32),
                'last':      torch.tensor(last,      dtype=torch.float32),
                'delta':     torch.tensor(delta,     dtype=torch.float32)}


class DS2D(Dataset):
    def __init__(self, event_dirs, sl=SEQ_LEN, ps=PRED_STEPS, stride=STRIDE, ns=None):
        self.sl, self.ps = sl, ps
        valid = []
        for ev in event_dirs:
            try:
                d = load_event(ev)
                if d['wl2d'].shape[0] > sl + ps + 1: valid.append(d)
            except Exception as e:
                print(f'  Skipping {ev.name}: {e}')
        self.rw = [d['wl2d'] for d in valid]
        self.rr = [d['rain'] for d in valid]
        if ns is None:
            all_wl = np.concatenate(self.rw).flatten()
            all_r  = np.concatenate(self.rr).flatten()
            self.ns = {'wm2': float(all_wl.mean()), 'ws2': float(max(all_wl.std(), 1.0)),
                       'rm':  float(all_r.mean()),  'rs':  float(max(all_r.std(), 1e-6))}
        else:
            self.ns = ns
        s = self.ns
        self.wn = [((w - s['wm2']) / s['ws2']).astype('float32') for w in self.rw]
        self.rn = [((r - s['rm'])  / s['rs']).astype('float32')  for r in self.rr]
        self.samples = [(i, t) for i, w in enumerate(self.wn)
                        for t in range(1, len(w) - sl - ps, stride)]
        print(f'  DS2D: {len(event_dirs)} Events ({len(valid)} valid) '
              f'-> {len(self.samples)} Samples')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        i, t = self.samples[idx]
        sl, ps, s = self.sl, self.ps, self.ns
        wl   = self.wn[i][t:t + sl]
        dwl  = wl - self.wn[i][t - 1:t + sl - 1]
        rain = self.rn[i][t:t + sl]
        cum_rain, rain_grad = _rain_extras(rain, s['rm'], s['rs'])
        last  = self.rw[i][t + sl - 1]
        delta = ((self.rw[i][t + sl:t + sl + ps] - last) / s['ws2']).astype('float32')
        return {'wl':        torch.tensor(wl,        dtype=torch.float32),
                'dwl':       torch.tensor(dwl,       dtype=torch.float32),
                'rain':      torch.tensor(rain,      dtype=torch.float32),
                'cum_rain':  torch.tensor(cum_rain,  dtype=torch.float32),
                'rain_grad': torch.tensor(rain_grad, dtype=torch.float32),
                'last':      torch.tensor(last,      dtype=torch.float32),
                'delta':     torch.tensor(delta,     dtype=torch.float32)}

print('Datasets defined.')

In [ ]:
# training utils
_AMP = (DEVICE == 'cuda')   # mixed precision only on gpu

def make_ema(model):
    ema = copy.deepcopy(model)
    for p in ema.parameters(): p.requires_grad_(False)
    for m in ema.modules():
        if hasattr(m, 'flatten_parameters'): m.flatten_parameters()
    return ema

def upd_ema(model, ema, decay=EMA_DECAY):
    with torch.no_grad():
        for p, ep in zip(model.parameters(), ema.parameters()):
            ep.mul_(decay).add_(p, alpha=1 - decay)

def fwd1d(model, batch, nf, ei, is_train):
    wl        = torch.nan_to_num(batch['wl'].to(DEVICE))
    dwl       = torch.nan_to_num(batch['dwl'].to(DEVICE))
    rain      = torch.nan_to_num(batch['rain'].to(DEVICE))
    cum_rain  = torch.nan_to_num(batch['cum_rain'].to(DEVICE))
    rain_grad = torch.nan_to_num(batch['rain_grad'].to(DEVICE))
    delta     = batch['delta'].to(DEVICE)
    if is_train:
        wl  = wl  + torch.randn_like(wl)  * NOISE_STD
        dwl = dwl + torch.randn_like(dwl) * NOISE_STD
    return model(wl, dwl, rain, cum_rain, rain_grad, nf, ei), delta

def fwd2d(model, batch, sf, ei2d, is_train):
    wl        = torch.nan_to_num(batch['wl'].to(DEVICE))
    dwl       = torch.nan_to_num(batch['dwl'].to(DEVICE))
    rain      = torch.nan_to_num(batch['rain'].to(DEVICE))
    cum_rain  = torch.nan_to_num(batch['cum_rain'].to(DEVICE))
    rain_grad = torch.nan_to_num(batch['rain_grad'].to(DEVICE))
    delta     = batch['delta'].to(DEVICE)
    if is_train:
        wl  = wl  + torch.randn_like(wl)  * NOISE_STD
        dwl = dwl + torch.randn_like(dwl) * NOISE_STD
    return model(wl, dwl, rain, cum_rain, rain_grad, sf, ei2d), delta

def train_model(model, ema, tr_ld, va_ld, fwd_fn, name, save_tag=None, arch_meta=None):
    opt    = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    sched  = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, 'min', patience=8, factor=0.5, min_lr=1e-6)
    scaler = torch.cuda.amp.GradScaler(enabled=_AMP)
    crit   = nn.HuberLoss(delta=1.0, reduction='none')
    step_w = torch.linspace(1.0, 0.6, PRED_STEPS).to(DEVICE)
    best       = float('inf')
    best_state = copy.deepcopy(ema.state_dict())
    pat        = 0
    hist       = {'tl': [], 'vl': [], 'rmse': [], 'lr': []}
    n_params   = sum(p.numel() for p in model.parameters())
    print(f'\n{"="*60}')
    print(f'  {name}  ({n_params:,} parameters)')
    print(f'{"="*60}')
    print(f'{"Ep":>4} {"Train":>12} {"Val":>12} {"RMSE":>12} {"LR":>10}')
    print('-' * 55)
    t0 = time.time()
    for ep in range(EPOCHS):
        model.train(); tl = 0; nb = 0
        for b in tr_ld:
            opt.zero_grad()
            with torch.amp.autocast(device_type=DEVICE, enabled=_AMP):
                p, d = fwd_fn(model, b, True)
                loss = (crit(p, d) * step_w[None, :, None]).mean()
            if torch.isnan(loss): continue
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update()
            upd_ema(model, ema)
            tl += loss.item(); nb += 1
        ema.eval(); vl_sum = 0; nv = 0; all_p, all_d = [], []
        with torch.no_grad():
            for b in va_ld:
                with torch.amp.autocast(device_type=DEVICE, enabled=_AMP):
                    p, d = fwd_fn(ema, b, False)
                v = (crit(p, d) * step_w[None, :, None]).mean()
                if not torch.isnan(v): vl_sum += v.item(); nv += 1
                all_p.append(p.cpu().numpy()); all_d.append(d.cpu().numpy())
        avg_tl = tl / max(nb, 1); avg_vl = vl_sum / max(nv, 1)
        rmse   = float(np.sqrt(np.nanmean(
            (np.concatenate(all_p) - np.concatenate(all_d)) ** 2
        ))) if all_p else float('inf')
        lr_now = opt.param_groups[0]['lr']
        sched.step(rmse)
        hist['tl'].append(avg_tl); hist['vl'].append(avg_vl)
        hist['rmse'].append(rmse); hist['lr'].append(lr_now)
        mark = ''; is_new_best = rmse < best
        if is_new_best:
            best = rmse; best_state = copy.deepcopy(ema.state_dict()); pat = 0; mark = '  *'
            if save_tag:
                torch.save(ema.state_dict(), SAVE_DIR / f'{save_tag}.pth')
                torch.save(ema.state_dict(), SAVE_DIR / f'{save_tag}_best.pth')
            if save_tag and ENABLE_KAGGLE_UPLOAD:
                _upload_checkpoint(f'{save_tag}_best', ep + 1, rmse, arch_meta)
        else:
            pat += 1
        if (save_tag and ENABLE_KAGGLE_UPLOAD and UPLOAD_EVERY_N_EPOCHS > 0
                and (ep + 1) % UPLOAD_EVERY_N_EPOCHS == 0 and not is_new_best):
            tmp_tag = f'{save_tag}_ep{ep+1}'
            torch.save(ema.state_dict(), SAVE_DIR / f'{tmp_tag}.pth')
            _upload_checkpoint(tmp_tag, ep + 1, rmse, arch_meta)
        print(f'{ep+1:4d} {avg_tl:12.8f} {avg_vl:12.8f} {rmse:12.8f} {lr_now:10.2e}{mark}')
        if pat >= PATIENCE: print('  Early stop.'); break
    elapsed = (time.time() - t0) / 60
    print(f'  Done in {elapsed:.1f} min | Best RMSE: {best:.8f}')
    ema.load_state_dict(best_state)
    for m in ema.modules():
        if hasattr(m, 'flatten_parameters'): m.flatten_parameters()
    return hist

print('Training utilities defined.')
print(f'  AMP (mixed precision): {"active" if _AMP else "inactive (no CUDA)"}')

In [ ]:
# evaluation metrics

def metric_rmse(p, a):
    m = ~(np.isnan(p) | np.isnan(a))
    return float(np.sqrt(np.mean((p[m] - a[m]) ** 2))) if m.sum() > 0 else float('nan')

def metric_r2(p, a):
    m = ~(np.isnan(p) | np.isnan(a))
    if m.sum() == 0: return float('nan')
    ss_res = np.sum((a[m] - p[m]) ** 2)
    ss_tot = np.sum((a[m] - a[m].mean()) ** 2)
    if ss_tot == 0: return 1.0 if ss_res == 0 else 0.0
    return float(1 - ss_res / ss_tot)

def metric_nse_per_node(p, a):
    N = p.shape[1]; vals = []
    for j in range(N):
        m = ~(np.isnan(p[:, j]) | np.isnan(a[:, j]))
        if m.sum() < 2: continue
        ss_r = np.sum((a[m, j] - p[m, j]) ** 2)
        ss_t = np.sum((a[m, j] - a[m, j].mean()) ** 2)
        vals.append(1 - ss_r / ss_t if ss_t > 0 else (1.0 if ss_r == 0 else 0.0))
    return float(np.mean(vals)) if vals else float('nan')

def metric_peak_rmse(p, a, pct=95):
    m = ~(np.isnan(p) | np.isnan(a))
    if m.sum() == 0: return float('nan')
    thr = np.percentile(a[m], pct)
    pm  = m & (a > thr)
    return float(np.sqrt(np.mean((p[pm] - a[pm]) ** 2))) if pm.sum() > 0 else float('nan')

def metric_rmse_over_time(p, a):
    return np.array([metric_rmse(p[t:t+1].flatten(), a[t:t+1].flatten())
                     for t in range(p.shape[0])])

def metric_mae(p, a):
    m = ~(np.isnan(p) | np.isnan(a))
    return float(np.mean(np.abs(p[m] - a[m]))) if m.sum() > 0 else float('nan')

def metric_kge(p, a):
    """kling-gupta efficiency (KGE=1 is perfect, >0.5 is good)"""
    m = ~(np.isnan(p) | np.isnan(a))
    if m.sum() < 2: return float('nan')
    r     = np.corrcoef(p[m], a[m])[0, 1]
    alpha = p[m].std() / (a[m].std() + 1e-8)
    beta  = p[m].mean() / (a[m].mean() + 1e-8)
    return float(1 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2))

def metric_srmse(p, a):
    """standardized RMSE = RMSE / std(observed), dimensionless and comparable across datasets
    SRMSE < 1 means model beats persistence; 0 = perfect"""
    m = ~(np.isnan(p) | np.isnan(a))
    if m.sum() == 0: return float('nan')
    rmse    = np.sqrt(np.mean((p[m] - a[m]) ** 2))
    std_obs = a[m].std()
    return float(rmse / std_obs) if std_obs > 1e-8 else float('nan')

def compute_all_metrics(p, a):
    return {'rmse':           metric_rmse(p.flatten(), a.flatten()),
            'mae':            metric_mae(p.flatten(), a.flatten()),
            'kge':            metric_kge(p.flatten(), a.flatten()),
            'r2':             metric_r2(p.flatten(), a.flatten()),
            'nse_per_node':   metric_nse_per_node(p, a),
            'peak_rmse':      metric_peak_rmse(p, a),
            'srmse':          metric_srmse(p.flatten(), a.flatten()),
            'rmse_over_time': metric_rmse_over_time(p, a)}

def aggregate_metrics(metrics_list):
    scalar_keys = ['rmse', 'mae', 'kge', 'r2', 'nse_per_node', 'peak_rmse', 'srmse']
    agg = {}
    for k in scalar_keys:
        vals = [m[k] for m in metrics_list if not np.isnan(m[k])]
        agg[k + '_mean'] = float(np.mean(vals)) if vals else float('nan')
        agg[k + '_std']  = float(np.std(vals))  if vals else float('nan')
    return agg

print('Metrics defined.')

In [ ]:
# autoregressive rollout

def rollout_event(ema_1d, ema_2d, ev_path, g1d, g2d, ei2d, ns1, ns2,
                  sl=SEQ_LEN, ps=PRED_STEPS, device=DEVICE):
    NF, EI, N1D = g1d
    SF, SDIM, N2D = g2d
    d1n = pd.read_csv(ev_path / '1d_nodes_dynamic_all.csv')
    d2n = pd.read_csv(ev_path / '2d_nodes_dynamic_all.csv')
    wl1d_df = d1n.pivot(index='timestep', columns='node_idx', values='water_level')
    wl2d_df = d2n.pivot(index='timestep', columns='node_idx', values='water_level')
    rain_s  = d2n.groupby('timestep')['rainfall'].mean().fillna(0)
    wl1d_df = wl1d_df.reindex(columns=range(N1D), fill_value=np.nan)
    wl2d_df = wl2d_df.reindex(columns=range(N2D), fill_value=np.nan)
    ts_int   = [int(t) for t in wl1d_df.index]
    ts_to_i  = {t: i for i, t in enumerate(ts_int)}
    T        = len(ts_int)
    rain_arr = rain_s.reindex(wl1d_df.index).fillna(0).values.astype('float32')
    known_mask = ~wl1d_df.isnull().any(axis=1).values
    n_known    = int(known_mask.sum())
    assert n_known > 0, f'No known timesteps in {ev_path}'
    # train split events: all timesteps known -> use first sl as context
    if n_known >= T:
        n_known = min(sl, T - ps)
    wl1d_abs  = wl1d_df.values.astype('float32').copy()
    wl2d_abs  = wl2d_df.values.astype('float32').copy()
    wl1d_true = wl1d_abs.copy(); wl2d_true = wl2d_abs.copy()
    for j in range(N1D):
        if np.all(np.isnan(wl1d_abs[:n_known, j])):
            wl1d_abs[:n_known, j] = np.nanmean(wl1d_abs[:n_known])
    for j in range(N2D):
        if np.all(np.isnan(wl2d_abs[:n_known, j])):
            wl2d_abs[:n_known, j] = np.nanmean(wl2d_abs[:n_known])
    def n1(a): return (a - ns1['wm'])  / ns1['ws']
    def n2(a): return (a - ns2['wm2']) / ns2['ws2']
    def nr(a): return (a - ns1['rm'])  / ns1['rs']
    wl1d_norm = np.zeros_like(wl1d_abs); wl2d_norm = np.zeros_like(wl2d_abs)
    rain_norm = nr(rain_arr)
    wl1d_norm[:n_known] = n1(wl1d_abs[:n_known])
    wl2d_norm[:n_known] = n2(wl2d_abs[:n_known])
    ema_1d.eval(); ema_2d.eval()
    def tt(a): return torch.tensor(a[None], dtype=torch.float32).to(device)
    t = n_known
    while t < T:
        cs  = max(0, t - sl)
        c1  = wl1d_norm[cs:t]; c2 = wl2d_norm[cs:t]; cr = rain_norm[cs:t]
        pad = sl - len(c1)
        if pad > 0:
            c1 = np.concatenate([np.tile(c1[[0]], (pad, 1)), c1], 0)
            c2 = np.concatenate([np.tile(c2[[0]], (pad, 1)), c2], 0)
            cr = np.concatenate([np.zeros(pad, 'float32'), cr])
        dw1   = c1 - np.concatenate([c1[[0]], c1[:-1]])
        dw2   = c2 - np.concatenate([c2[[0]], c2[:-1]])
        cum_r = (np.cumsum(cr) / np.arange(1, sl + 1, dtype='float32')).astype('float32')
        grad_r = (np.gradient(cr) / (ns1['rs'] + 1e-8)).astype('float32')
        steps = min(ps, T - t)
        with torch.no_grad():
            d1 = ema_1d(tt(c1), tt(dw1), tt(cr), tt(cum_r), tt(grad_r), NF, EI)[0].cpu().numpy()
            d2 = ema_2d(tt(c2), tt(dw2), tt(cr), tt(cum_r), tt(grad_r), SF, ei2d)[0].cpu().numpy()
        p1 = wl1d_abs[t-1][None] + d1[:steps] * ns1['ws']
        p2 = wl2d_abs[t-1][None] + d2[:steps] * ns2['ws2']
        wl1d_abs[t:t+steps]  = p1; wl2d_abs[t:t+steps]  = p2
        wl1d_norm[t:t+steps] = n1(p1); wl2d_norm[t:t+steps] = n2(p2)
        t += steps
    return wl1d_abs, wl2d_abs, wl1d_true, wl2d_true, n_known, ts_to_i

print('Rollout defined.')

In [ ]:
# evaluation loop and plotting

def evaluate_model(model_name, ema_1d, ema_2d, test_dirs, graphs, ns_dict):
    g1d, g2d, ei2d = graphs['1d'], graphs['2d'], graphs['2d_ei']
    ns1, ns2       = ns_dict['1d'], ns_dict['2d']
    results = {}
    for node_type in ['1d', '2d']:
        metrics_list = []; rot_list = []
        for ev in test_dirs:
            try:
                wl1d_p, wl2d_p, wl1d_t, wl2d_t, n_known, _ = \
                    rollout_event(ema_1d, ema_2d, ev, g1d, g2d, ei2d, ns1, ns2)
                p = wl1d_p[n_known:] if node_type == '1d' else wl2d_p[n_known:]
                a = wl1d_t[n_known:] if node_type == '1d' else wl2d_t[n_known:]
                if np.all(np.isnan(a)): continue
                m = compute_all_metrics(p, a)
                metrics_list.append(m); rot_list.append(m['rmse_over_time'])
            except Exception as ex:
                print(f'    Skip {ev.name}: {ex}')
        if metrics_list:
            agg    = aggregate_metrics(metrics_list)
            maxlen = max(len(r) for r in rot_list)
            padded = [np.pad(r, (0, maxlen - len(r)), constant_values=np.nan) for r in rot_list]
            agg['rmse_over_time'] = np.nanmean(padded, axis=0)
            results[node_type]    = agg
            print(f'  {model_name} {node_type.upper()}:')
            print(f'    RMSE     = {agg["rmse_mean"]:.8f} +- {agg["rmse_std"]:.8f}')
            print(f'    MAE      = {agg["mae_mean"]:.8f} +- {agg["mae_std"]:.8f}')
            print(f'    KGE      = {agg["kge_mean"]:.8f} +- {agg["kge_std"]:.8f}')
            print(f'    R2       = {agg["r2_mean"]:.8f} +- {agg["r2_std"]:.8f}')
            print(f'    NSE/node = {agg["nse_per_node_mean"]:.8f} +- {agg["nse_per_node_std"]:.8f}')
            print(f'    PeakRMSE = {agg["peak_rmse_mean"]:.8f} +- {agg["peak_rmse_std"]:.8f}')
            print(f'    SRMSE    = {agg["srmse_mean"]:.8f} +- {agg["srmse_std"]:.8f}')
    return results

def print_comparison_table(all_results):
    hdr = f'{"Model":<25} {"Type":<5} {"RMSE (mean+-std)":>22} {"MAE":>22} {"KGE":>22} {"R2":>22} {"NSE/node":>22} {"PeakRMSE":>22} {"SRMSE":>22}'
    print(f'\n{"="*len(hdr)}')
    print(hdr)
    print(f'{"-"*len(hdr)}')
    for model_name, res in all_results.items():
        for nt, agg in res.items():
            print(f'{model_name:<25} {nt.upper():<5} '
                  f'{agg["rmse_mean"]:>9.8f}+-{agg["rmse_std"]:.8f} '
                  f'{agg["mae_mean"]:>9.8f}+-{agg["mae_std"]:.8f} '
                  f'{agg["kge_mean"]:>9.8f}+-{agg["kge_std"]:.8f} '
                  f'{agg["r2_mean"]:>9.8f}+-{agg["r2_std"]:.8f} '
                  f'{agg["nse_per_node_mean"]:>9.8f}+-{agg["nse_per_node_std"]:.8f} '
                  f'{agg["peak_rmse_mean"]:>9.8f}+-{agg["peak_rmse_std"]:.8f} '
                  f'{agg["srmse_mean"]:>9.8f}+-{agg["srmse_std"]:.8f}')
    print(f'{"="*len(hdr)}')

def plot_rmse_over_time(all_results, title, save_path=None):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, nt in zip(axes, ['1d', '2d']):
        for model_name, res in all_results.items():
            if nt in res and 'rmse_over_time' in res[nt]:
                ax.plot(res[nt]['rmse_over_time'], label=model_name, linewidth=2)
        ax.set_xlabel('Autoregressive step'); ax.set_ylabel('RMSE')
        ax.set_title(f'{nt.upper()} Nodes'); ax.legend(); ax.grid(alpha=0.3)
    plt.suptitle(title, fontsize=13); plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150); print(f'Plot saved: {save_path}')
    plt.show()

def save_results_json(all_results, path):
    save = {}
    for model_name, res in all_results.items():
        save[model_name] = {}
        for nt, agg in res.items():
            save[model_name][nt] = {
                k: v.tolist() if isinstance(v, np.ndarray) else float(v)
                for k, v in agg.items()}
    with open(path, 'w') as f: _json.dump(save, f, indent=2)
    print(f'Results saved: {path}')

print('Evaluation loop defined.')
print('\n  Shared code fully loaded.')
print('  Next step: model architecture.')

---
## Model architecture: GAT + LSTM

In [ ]:
# model architecture - GAT + LSTM

class GAT_LSTM(nn.Module):
    """2-layer GAT (fixed) + LSTM temporal. same for 1D and 2D"""
    def __init__(self, static_dim, h_gnn=H_GNN, h_dim=H_DIM, heads=4, drop=DROP):
        super().__init__()
        self.static_enc = nn.Linear(static_dim, h_gnn)
        # 2-layer GAT w/ skip connection (from GNN_v11_eval)
        self.gat1  = GATConv(2 + h_gnn, h_gnn, heads=heads, dropout=drop, concat=False)
        self.gat2  = GATConv(h_gnn,     h_gnn, heads=heads, dropout=drop, concat=False)
        self.norm1 = nn.LayerNorm(h_gnn)
        self.norm2 = nn.LayerNorm(h_gnn)
        self.skip  = nn.Linear(2 + h_gnn, h_gnn, bias=False)
        self.lstm = nn.LSTM(h_gnn + 3, h_dim, num_layers=2, batch_first=True, dropout=drop)
        self.head = nn.Sequential(nn.LayerNorm(h_dim), nn.Linear(h_dim, PRED_STEPS))
        self.drop = nn.Dropout(drop)

    def forward(self, wl, dwl, rain, cum_rain, rain_grad, nf, ei):
        B, T, N = wl.shape
        se = F.elu(self.static_enc(nf))                                    # (N, h_gnn)

        # stack all T timesteps into one big batch (B*T subgraphs)
        # mathematically same as looping over T but single GAT call
        # = better gpu usage, no pytohn loop overhead
        nt     = torch.stack([wl, dwl], dim=-1).reshape(B*T*N, 2)          # (B*T*N, 2)
        se_exp = se.unsqueeze(0).expand(B*T, -1, -1).reshape(B*T*N, -1)    # (B*T*N, h_gnn)
        offsets = torch.arange(B*T, device=ei.device) * N                  # (B*T,)
        ei_bt   = (ei[:, None, :] + offsets[None, :, None]).reshape(2, -1) # (2, B*T*E)

        xf = torch.cat([nt, se_exp], dim=-1)                                # (B*T*N, 2+h_gnn)
        sk = self.skip(xf)                                                  # (B*T*N, h_gnn)
        h  = F.elu(self.norm1(self.gat1(xf, ei_bt)))                       # (B*T*N, h_gnn)
        h  = F.elu(self.norm2(self.gat2(h,  ei_bt) + sk))                  # (B*T*N, h_gnn)
        h  = self.drop(h)

        seq = h.reshape(B, T, N, -1)                                        # (B, T, N, h_gnn)
        rf  = torch.stack([rain, cum_rain, rain_grad], dim=-1).unsqueeze(2).expand(-1, -1, N, -1)
        seq = torch.cat([seq, rf], dim=-1).permute(0, 2, 1, 3).reshape(B*N, T, -1)

        out, _ = self.lstm(seq)
        delta  = self.head(self.drop(out[:, -1, :]))
        return delta.reshape(B, N, PRED_STEPS).permute(0, 2, 1)


print('GAT_LSTM defined.')
print(f'  Spatial:  2-Layer GAT + Skip (heads=4, h_gnn={H_GNN}) -- vectorized over T')
print(f'  Temporal: LSTM (num_layers=2, h_dim={H_DIM})')

---
## Load data & create datasets

In [ ]:
# load graph data and splits
print('Loading graph data...')
(NF_1D, EI_1D, N_1D), (SF_2D, SDIM_2D, N_2D), EI_2D = load_graph_data(MODEL_PATH, DEVICE)

print('\nSplits:')
train_dirs, val_dirs, test_dirs = get_splits(MODEL_PATH)
print(f'  Test events: {[d.name for d in test_dirs]}')

In [ ]:
# set up datasets and dataloaders
_NW = 2   # num_workers for parallel data loading (cpu overlaps w/ gpu training)

print('1D datasets...')
tr_ds1 = DS1D(train_dirs)
va_ds1 = DS1D(val_dirs,   ns=tr_ds1.ns)
te_ds1 = DS1D(test_dirs,  ns=tr_ds1.ns)

tr_ld1 = DataLoader(tr_ds1, BS_1D, shuffle=True,  drop_last=True,
                    num_workers=_NW, persistent_workers=True,
                    pin_memory=(DEVICE == 'cuda'))
va_ld1 = DataLoader(va_ds1, BS_1D, shuffle=False, drop_last=False,
                    num_workers=_NW, persistent_workers=True,
                    pin_memory=(DEVICE == 'cuda'))

print('\n2D datasets...')
tr_ds2 = DS2D(train_dirs)
va_ds2 = DS2D(val_dirs,   ns=tr_ds2.ns)
te_ds2 = DS2D(test_dirs,  ns=tr_ds2.ns)

tr_ld2 = DataLoader(tr_ds2, BS_2D, shuffle=True,  drop_last=True,
                    num_workers=_NW, persistent_workers=True,
                    pin_memory=(DEVICE == 'cuda'))
va_ld2 = DataLoader(va_ds2, BS_2D, shuffle=False, drop_last=False,
                    num_workers=_NW, persistent_workers=True,
                    pin_memory=(DEVICE == 'cuda'))

NS_1D = tr_ds1.ns
NS_2D = tr_ds2.ns
print(f'\n  1D normalization: wm={NS_1D["wm"]:.4f}, ws={NS_1D["ws"]:.4f}')
print(f'  2D normalization: wm={NS_2D["wm2"]:.4f}, ws={NS_2D["ws2"]:.4f}')

---
## Training

In [ ]:
# train 1D network
model_1d = GAT_LSTM(NF_1D.shape[1]).to(DEVICE)

# forget gate bias init to 1.0 (lstm-specific, helps gradients early on)
# pytorch stores lstm biases as [i, f, g, o] * hidden_size -> forget = index [n:2n]
for name, param in model_1d.lstm.named_parameters():
    if 'bias' in name:
        n = param.shape[0] // 4
        nn.init.ones_(param[n:2*n])
print('  Forget gate bias initialized (1D)')

ema_1d   = make_ema(model_1d)

fwd1d_fn = lambda m, b, tr: fwd1d(m, b, NF_1D, EI_1D, tr)

hist_1d = train_model(
    model_1d, ema_1d, tr_ld1, va_ld1, fwd1d_fn,
    name      = 'GAT+LSTM 1D',
    save_tag  = 'GAT_LSTM_1d',
    arch_meta = {'spatial': 'GAT', 'temporal': 'LSTM', 'dim': '1D'},
)


# train 2D network
model_2d = GAT_LSTM(SDIM_2D).to(DEVICE)

# forget gate bias init to 1.0
for name, param in model_2d.lstm.named_parameters():
    if 'bias' in name:
        n = param.shape[0] // 4
        nn.init.ones_(param[n:2*n])
print('  Forget gate bias initialized (2D)')

ema_2d   = make_ema(model_2d)

fwd2d_fn = lambda m, b, tr: fwd2d(m, b, SF_2D, EI_2D, tr)

hist_2d = train_model(
    model_2d, ema_2d, tr_ld2, va_ld2, fwd2d_fn,
    name      = 'GAT+LSTM 2D',
    save_tag  = 'GAT_LSTM_2d',
    arch_meta = {'spatial': 'GAT', 'temporal': 'LSTM', 'dim': '2D'},
)

---
## Evaluation

In [ ]:
# evaluate on test set
graphs  = {'1d': (NF_1D, EI_1D, N_1D), '2d': (SF_2D, SDIM_2D, N_2D), '2d_ei': EI_2D}
ns_dict = {'1d': NS_1D, '2d': NS_2D}

all_results = {}
all_results['GAT+LSTM'] = evaluate_model('GAT+LSTM', ema_1d, ema_2d,
                                            test_dirs, graphs, ns_dict)

print_comparison_table(all_results)

plot_rmse_over_time(
    all_results,
    title     = 'GAT+LSTM: RMSE over autoregressive timesteps',
    save_path = SAVE_DIR / 'rmse_over_time_lstm.png',
)

save_results_json(all_results, SAVE_DIR / 'results_GAT_LSTM.json')